# Pokemon: esegui le ricette Python

Il codice dei modelli e su GitHub; qui servono solo configurazione, download e avvio.
Attualmente e disponibile **1 ricetta su 3**. Le altre saranno aggiunte a `run_all.py`.

Prima di eseguire: aggiungi il dataset, attiva **Internet** e configura il Secret
**GITHUB_TOKEN** con accesso in lettura al repository privato `Jackfnvpn/HackathonPokemon`.
Abilita il Secret per questo notebook; non incollare il token nel codice.
Non serve una GPU. Lo score 0.69000 del primo modello e storico.


In [ ]:
REPOSITORY = "Jackfnvpn/HackathonPokemon"
REF = "main"  # Per ripetere una versione precisa, usa il suo SHA di commit.
DATA_DIR = None  # Oppure: "/kaggle/input/nome-dataset/cartella"
OUTPUT_DIR = "/kaggle/working/outputs"
RUN_CV = False
SAVE_MODELS = False


In [ ]:
import io
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import zipfile
from kaggle_secrets import UserSecretsClient

request = Request(
    f"https://api.github.com/repos/{REPOSITORY}/zipball/{REF}",
    headers={"Authorization": "Bearer " + UserSecretsClient().get_secret("GITHUB_TOKEN"),
             "Accept": "application/vnd.github+json"},
)
try:
    with urlopen(request, timeout=120) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
except HTTPError as error:
    raise RuntimeError(f"Download GitHub fallito (HTTP {error.code}). Controlla Secret e accesso al repository.") from None
finally:
    del request
code_directory = Path("/kaggle/working/github_code").resolve()
code_directory.mkdir(parents=True, exist_ok=True)
with archive:
    for member in archive.namelist():
        if code_directory not in (code_directory / member).resolve().parents:
            raise ValueError("Percorso non valido nell'archivio GitHub.")
    repository_folder = archive.namelist()[0].split("/")[0]
    archive.extractall(code_directory)
REPO_DIR = code_directory / repository_folder
print("Codice:", REPO_DIR)


In [ ]:
import subprocess
import sys

command = [sys.executable, "-u", str(REPO_DIR / "run_all.py"), "--output-dir", OUTPUT_DIR]
if DATA_DIR is not None:
    command += ["--data-dir", str(DATA_DIR)]
if RUN_CV:
    command.append("--cv")
if SAVE_MODELS:
    command.append("--save-models")
subprocess.run(command, check=True)


In [ ]:
from IPython.display import FileLink, display

for csv_path in sorted(Path(OUTPUT_DIR).glob("*/submission*.csv")):
    print(csv_path.parent.name)
    display(FileLink(str(csv_path)))
